In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
# =========================
# Test your saved VAD model
# =========================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "/content/drive/MyDrive/Thesis docs/models/Filipa-model/checkpoint-1008"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loaded model from: /content/drive/MyDrive/Thesis docs/models/Filipa-model/checkpoint-504 (1)
Problem type: regression


In [12]:
def predict_vad(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    valence, arousal, dominance = outputs.logits[0].cpu().numpy()

    return {
        "text": text,
        "valence": float(valence),
        "arousal": float(arousal),
        "dominance": float(dominance)
    }

In [21]:
predict_vad("I feel happy, safe, and excited about the future.")

{'text': 'I feel happy, safe, and excited about the future.',
 'valence': 3.1610255241394043,
 'arousal': 3.01688289642334,
 'dominance': 3.1416454315185547}

In [22]:
predict_vad("I feel empty, hopeless, and completely exhausted.")

{'text': 'I feel empty, hopeless, and completely exhausted.',
 'valence': 3.1391892433166504,
 'arousal': 2.9997811317443848,
 'dominance': 3.132180690765381}

In [23]:
predict_vad("The package was delivered on Tuesday afternoon.")

{'text': 'The package was delivered on Tuesday afternoon.',
 'valence': 3.1391892433166504,
 'arousal': 2.9997811317443848,
 'dominance': 3.132180690765381}

In [24]:
texts = [
    "I am extremely happy and excited about my life.",
    "I feel hopeless, empty, and completely exhausted.",
    "The package was delivered on Tuesday afternoon.",
    "I am furious and shaking with anger.",
]

inputs = tokenizer(
    texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
).to(device)

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.logits)

tensor([[3.1610, 3.0169, 3.1416],
        [3.1392, 2.9998, 3.1322],
        [3.1392, 2.9998, 3.1322],
        [3.1392, 2.9998, 3.1322]])


In [25]:
texts = [
    "I won the lottery today and I have never been happier in my life.",
    "I am crying because everything feels hopeless and meaningless.",
    "The weather forecast says it will rain tomorrow.",
    "I am absolutely furious and want to scream at everyone.",
    "I feel calm and relaxed while reading a book."
]

inputs = tokenizer(
    texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
).to(device)

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

for text, pred in zip(texts, outputs.logits.cpu().numpy()):
    print("-" * 60)
    print(text)
    print(pred)

------------------------------------------------------------
I won the lottery today and I have never been happier in my life.
[3.2010427 3.013783  3.1405234]
------------------------------------------------------------
I am crying because everything feels hopeless and meaningless.
[3.1610253 3.016883  3.141645 ]
------------------------------------------------------------
The weather forecast says it will rain tomorrow.
[3.1523013 3.011883  3.1388366]
------------------------------------------------------------
I am absolutely furious and want to scream at everyone.
[3.1688228 3.0201335 3.1420302]
------------------------------------------------------------
I feel calm and relaxed while reading a book.
[3.1610255 3.0168831 3.141645 ]
